In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import joblib

In [2]:
df = pd.read_csv("C:/Manasi/NMIMS/Manasi/Capstone Project/Agriculture/Datasets/agri.csv")
df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_x0020_Price,Max_x0020_Price,Modal_x0020_Price
0,Andhra Pradesh,East Godavari,Ravulapelem,Banana,Chakkarakeli(Red),Large,25-10-24,2100.0,2600.0,2400.0
1,Andhra Pradesh,East Godavari,Ravulapelem,Banana,Karpura,Large,25-10-24,1200.0,2400.0,1800.0
2,Assam,Barpeta,Barpeta Road,Green Chilli,Green Chilly,Local,25-10-24,8000.0,9000.0,9000.0
3,Assam,Goalpara,Lakhipur,Pumpkin,Pumpkin,Local,25-10-24,1500.0,2000.0,2000.0
4,Assam,Goalpara,Lakhipur,Tomato,Tomato,Local,25-10-24,3000.0,3500.0,3400.0


In [3]:
df = df.rename(columns={
    'Min_x0020_Price': 'Min_Price',
    'Max_x0020_Price': 'Max_Price',
    'Modal_x0020_Price': 'Modal_Price'
})

df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price
0,Andhra Pradesh,East Godavari,Ravulapelem,Banana,Chakkarakeli(Red),Large,25-10-24,2100.0,2600.0,2400.0
1,Andhra Pradesh,East Godavari,Ravulapelem,Banana,Karpura,Large,25-10-24,1200.0,2400.0,1800.0
2,Assam,Barpeta,Barpeta Road,Green Chilli,Green Chilly,Local,25-10-24,8000.0,9000.0,9000.0
3,Assam,Goalpara,Lakhipur,Pumpkin,Pumpkin,Local,25-10-24,1500.0,2000.0,2000.0
4,Assam,Goalpara,Lakhipur,Tomato,Tomato,Local,25-10-24,3000.0,3500.0,3400.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11274 entries, 0 to 11273
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   State         11274 non-null  object 
 1   District      11274 non-null  object 
 2   Market        11274 non-null  object 
 3   Commodity     11274 non-null  object 
 4   Variety       11274 non-null  object 
 5   Grade         11274 non-null  object 
 6   Arrival_Date  11274 non-null  object 
 7   Min_Price     11274 non-null  float64
 8   Max_Price     11274 non-null  float64
 9   Modal_Price   11274 non-null  float64
dtypes: float64(3), object(7)
memory usage: 880.9+ KB


In [5]:
df['Arrival_Date'] = pd.to_datetime(df['Arrival_Date'], format='%d-%m-%y')

In [6]:
for column in df.columns:
    print(f"Unique values in '{column}':")
    print(df[column].unique(), "\n")

Unique values in 'State':
['Andhra Pradesh' 'Assam' 'Bihar' 'Chattisgarh' 'Gujarat' 'Haryana'
 'Himachal Pradesh' 'Jammu and Kashmir' 'Karnataka' 'Kerala'
 'Madhya Pradesh' 'Maharashtra' 'Manipur' 'Meghalaya' 'Nagaland' 'Odisha'
 'Punjab' 'Rajasthan' 'Tamil Nadu' 'Telangana' 'Tripura' 'Uttar Pradesh'
 'Uttrakhand' 'West Bengal' 'Goa'] 

Unique values in 'District':
['East Godavari' 'Barpeta' 'Goalpara' 'Kamrup' 'Karbi Anglong' 'MORIGAON'
 'Nalbari' 'Sibsagar' 'Sonitpur' 'Buxar' 'Jehanabad' 'Rohtas'
 'Balodabazar' 'Bastar' 'Dhamtari' 'Durg' 'Jashpur' 'Kanker' 'Korba'
 'Mahasamund' 'Raigarh' 'Rajnandgaon' 'Surajpur' 'Amreli' 'Anand'
 'Bharuch' 'Dahod' 'Gir Somnath' 'Junagarh' 'Kheda' 'Navsari' 'Rajkot'
 'Sabarkantha' 'Surendranagar' 'Vadodara(Baroda)' 'Ambala' 'Bhiwani'
 'Fatehabad' 'Gurgaon' 'Hissar' 'Jind' 'Kurukshetra'
 'Mahendragarh-Narnaul' 'Mewat' 'Panipat' 'Rewari' 'Sirsa' 'Sonipat'
 'Yamuna Nagar' 'Chamba' 'Hamirpur' 'Kangra' 'Kullu' 'Mandi' 'Anantnag'
 'Badgam' 'Jammu' 'Kathua' 

In [7]:
df.describe()

,Arrival_Date,Min_Price,Max_Price,Modal_Price
count,11274,11274.000000,11274.000000,11274.000000
mean,2024-10-25 00:00:00,4925.982394,5673.444135,5515.991921
min,2024-10-25 00:00:00,0.000000,0.000000,5.000000
25%,2024-10-25 00:00:00,2500.000000,3000.000000,3000.000000
50%,2024-10-25 00:00:00,4000.000000,4450.000000,4200.000000
75%,2024-10-25 00:00:00,5500.000000,6000.000000,6000.000000
max,2024-10-25 00:00:00,198000.000000,198080.000000,198040.000000
std,NaN,5282.789940,5879.686480,5703.219799


In [8]:
df.isnull().sum()

State           0
District        0
Market          0
Commodity       0
Variety         0
Grade           0
Arrival_Date    0
Min_Price       0
Max_Price       0
Modal_Price     0
dtype: int64

In [9]:
df.sort_values(by='Arrival_Date', inplace=True)

In [10]:
def outlier_filter(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

In [11]:
# Apply outlier filtering for all price columns and store the result back into df
df_filtered = outlier_filter(df, 'Min_Price')
df_filtered = outlier_filter(df_filtered, 'Max_Price')
df_filtered = outlier_filter(df_filtered, 'Modal_Price')

In [12]:
df_encoded = df_filtered.copy()

In [13]:
categorical_columns = ['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade']

In [14]:
label_encoders = {}
for column in categorical_columns:
    le = LabelEncoder()
    df_encoded[column] = le.fit_transform(df_encoded[column])
    label_encoders[column] = le

In [15]:
X = df_encoded.drop(columns=['Modal_Price', 'Arrival_Date'])  # Features (excluding target and date)
y = df_encoded['Modal_Price']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
X_train.head()

,State,District,Market,Commodity,Variety,Grade,Min_Price,Max_Price
2084,21,182,193,57,112,0,4300.0,4500.0
9181,6,143,287,22,191,0,2500.0,3000.0
8460,18,293,675,11,32,2,2500.0,2500.0
4191,19,240,413,118,0,0,3700.0,3700.0
184,6,143,288,28,62,0,5800.0,6000.0


# Random Forest Regressor

In [18]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

In [19]:
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [20]:
y_pred = rf_model.predict(X_test)

In [21]:
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # Root Mean Squared Error
r2 = r2_score(y_test, y_pred)

c:\Manasi\NMIMS\Manasi\Capstone Project\Agriculture\Bhoomi\venv\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [22]:
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R² Score: {r2}")

Mean Squared Error: 12486.705843141764
Root Mean Squared Error: 111.74392978207705
R² Score: 0.9962960756680224


# Hypertuned Random Forest

In [23]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

In [24]:
rf = RandomForestRegressor(random_state=42)

In [25]:
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

In [26]:
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 324 candidates, totalling 972 fits


c:\Manasi\NMIMS\Manasi\Capstone Project\Agriculture\Bhoomi\venv\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
324 fits failed out of a total of 972.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
141 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Manasi\NMIMS\Manasi\Capstone Project\Agriculture\Bhoomi\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Manasi\NMIMS\Manasi\Capstone Project\Agriculture\Bhoomi\venv\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "c:\Manasi\NMIMS\Manasi\Capstone Project\Agriculture\Bhoomi

GridSearchCV(cv=3, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20, 30, None],
                         'max_features': ['auto', 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='neg_mean_squared_error', verbose=2)

In [27]:
best_params = grid_search.best_params_
print("Best Hyperparameters: ", best_params)

Best Hyperparameters:  {'max_depth': 30, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


In [28]:
best_rf_model = grid_search.best_estimator_

In [29]:
y_pred_best = best_rf_model.predict(X_test)

In [30]:
mse_best = mean_squared_error(y_test, y_pred_best)
rmse_best = mean_squared_error(y_test, y_pred_best, squared=False)
r2_best = r2_score(y_test, y_pred_best)

c:\Manasi\NMIMS\Manasi\Capstone Project\Agriculture\Bhoomi\venv\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [31]:
print(f"Best Mean Squared Error: {mse_best}")
print(f"Best Root Mean Squared Error: {rmse_best}")
print(f"Best R² Score: {r2_best}")

Best Mean Squared Error: 11910.029000338684
Best Root Mean Squared Error: 109.1330793130052
Best R² Score: 0.9964671349863549


# XGBoost

In [32]:
xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=10, random_state=42)

In [33]:
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=300, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [34]:
y_pred_xgb = xgb_model.predict(X_test)

In [35]:
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = mean_squared_error(y_test, y_pred_xgb, squared=False)  # Root Mean Squared Error
r2_xgb = r2_score(y_test, y_pred_xgb)

c:\Manasi\NMIMS\Manasi\Capstone Project\Agriculture\Bhoomi\venv\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [36]:
print(f"XGBoost Mean Squared Error: {mse_xgb}")
print(f"XGBoost Root Mean Squared Error: {rmse_xgb}")
print(f"XGBoost R² Score: {r2_xgb}")

XGBoost Mean Squared Error: 11190.599180894145
XGBoost Root Mean Squared Error: 105.78562842321325
XGBoost R² Score: 0.9966805390375806


# Hypertuned XGBoost

In [37]:
param_grid_xgb = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [5, 10, 15],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

In [38]:
xgb = XGBRegressor(random_state=42)

In [39]:
grid_search_xgb = GridSearchCV(estimator=xgb, param_grid=param_grid_xgb, cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

In [ ]:
grid_search_xgb.fit(X_train, y_train)

Fitting 3 folds for each of 729 candidates, totalling 2187 fits


In [ ]:
best_params_xgb = grid_search_xgb.best_params_
print("Best Hyperparameters for XGBoost: ", best_params_xgb)

In [ ]:
best_xgb_model = grid_search_xgb.best_estimator_

In [ ]:
y_pred_best_xgb = best_xgb_model.predict(X_test)

In [ ]:
mse_best_xgb = mean_squared_error(y_test, y_pred_best_xgb)
rmse_best_xgb = mean_squared_error(y_test, y_pred_best_xgb, squared=False)
r2_best_xgb = r2_score(y_test, y_pred_best_xgb)

In [ ]:
print(f"Best XGBoost Mean Squared Error: {mse_best_xgb}")
print(f"Best XGBoost Root Mean Squared Error: {rmse_best_xgb}")
print(f"Best XGBoost R² Score: {r2_best_xgb}")

# Save the model

In [ ]:
joblib.dump(best_xgb_model, 'xgboost_model.pkl')